# M3DC1 Surrogate Model Training

This notebook trains surrogate models to predict growth rate (gamma) from:
- **Equilibria parameters**: R0, a, kappa, delta
- **Profile characteristics**: q0, q95, qmin, p0

**Models**: Random Forest Regressor (RFR) and Multi-Layer Perceptron (MLP)

**Dataset**: Curated dataset from `sdata03.h5` (9,891 complete cases)

In [1]:
# Imports - Critical packages only
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
import sys
import warnings
import types

# Suppress warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*numpy.dtype size changed.*')
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Mock optional dependencies to avoid NumPy/TensorFlow compatibility issues
mock_gpflow = types.ModuleType('gpflow')
mock_gpflow.__version__ = "0.0.0"
sys.modules['gpflow'] = mock_gpflow
sys.modules['tensorflow'] = types.ModuleType('tensorflow')

# Add SURGE to path and import
surge_src_path = Path('/global/homes/a/asvillar/src/SURGE')
if str(surge_src_path) not in sys.path:
    sys.path.insert(0, str(surge_src_path))

from surge import SurrogateEngine

# Configure matplotlib
plt.ioff()
matplotlib.rcParams['figure.max_open_warning'] = 0
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams['mathtext.fontset'] = 'dejavusans'

## 1. Load Curated Dataset

Load the curated (complete) dataset for surrogate model training. 
If the curated dataset file doesn't exist, this notebook will attempt to load from `sdata03.h5` and create it.


In [2]:
# Load curated dataset for surrogate model training
print("=" * 80)
print("LOADING CURATED DATASET")
print("=" * 80)

output_dir = Path('/pscratch/sd/a/asvillar/mp288/jobs/batch_16')
curated_file_pkl = output_dir / 'sdata03_curated.pkl'
curated_file_csv = output_dir / 'sdata03_curated.csv'

# Try to load curated dataset from pickle file
if curated_file_pkl.exists():
    print(f"\n✅ Loading curated dataset from pickle file...")
    df_complete = pd.read_pickle(curated_file_pkl)
    print(f"   File: {curated_file_pkl}")
    print(f"   Shape: {df_complete.shape}")
    print(f"   Memory usage: {df_complete.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print("\n✅ Curated dataset loaded successfully!")
else:
    print(f"\n⚠️  Curated dataset file not found: {curated_file_pkl}")
    print("\nAttempting to load from HDF5 and create curated dataset...")
    
    # Try to load from HDF5 and create curated dataset
    try:
        import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent.parent / "scripts" / "m3dc1"))
from loader import convert_to_dataframe
        
        hdf5_file = output_dir / 'sdata03.h5'
        if hdf5_file.exists():
            print(f"\n✅ Loading from HDF5: {hdf5_file}")
            df = convert_to_dataframe(hdf5_file)
            print(f"   Loaded DataFrame shape: {df.shape}")
            
            # Define critical variables for data curation
            critical_vars = {
                'gamma': ['output_gamma', 'gamma'],
                'q0': ['q0'],
                'q95': ['q95'],
                'qmin': ['output_qmin', 'qmin'],
                'p0': ['p0'],
                'R0': ['eq_R0', 'R0'],
                'a': ['eq_a', 'a'],
                'delta': ['eq_delta', 'delta'],
                'kappa': ['eq_kappa', 'kappa']
            }
            
            # Find actual column names
            actual_cols = {}
            for var, alternatives in critical_vars.items():
                for alt in alternatives:
                    if alt in df.columns:
                        actual_cols[var] = alt
                        break
            
            # Check for missing variables
            print("\nChecking data completeness...")
            complete_mask = pd.Series([True] * len(df), index=df.index)
            for var, col in actual_cols.items():
                if col in df.columns:
                    complete_mask = complete_mask & df[col].notna()
            
            df_complete = df[complete_mask].copy()
            
            # Also check HDF5 for profile completeness
            excluded_run_ids = []
            import h5py
            with h5py.File(hdf5_file, 'r') as f:
                run_keys = sorted([k for k in f.keys() if k.startswith('run_')], 
                                  key=lambda x: int(x.split('_')[1]))
                
                for run_key in run_keys:
                    excluded = False
                    # Check for missing profiles
                    if 'output' in f[run_key]:
                        output_group = f[run_key]['output']
                        if 'q' not in output_group or 'p' not in output_group:
                            excluded = True
                        elif output_group['p'] is not None:
                            p_data = output_group['p'][:]
                            if len(p_data) > 0 and np.any(p_data[:, 1] < 0):
                                excluded = True
                    
                    if excluded:
                        excluded_run_ids.append(run_key)
            
            # Remove excluded runs
            if 'run_id' in df_complete.columns and len(excluded_run_ids) > 0:
                df_complete = df_complete[~df_complete['run_id'].isin(excluded_run_ids)].copy()
            
            print(f"✅ Created curated dataset: {len(df_complete):,} cases")
            print(f"   Excluded: {len(df) - len(df_complete):,} cases")
            
            # Save for future use
            df_complete.to_pickle(curated_file_pkl)
            print(f"\n✅ Saved curated dataset to: {curated_file_pkl}")
            
        else:
            print(f"❌ HDF5 file not found: {hdf5_file}")
            print("   Please ensure either:")
            print(f"   1. The curated dataset exists at: {curated_file_pkl}")
            print(f"   2. Or the HDF5 file exists at: {hdf5_file}")
            raise FileNotFoundError("Cannot find curated dataset or HDF5 file")
            
    except Exception as e:
        print(f"❌ Error creating curated dataset: {e}")
        import traceback
        traceback.print_exc()
        raise

print("\n" + "=" * 80)
print("CURATED DATASET SUMMARY")
print("=" * 80)
print(f"\nTotal cases in curated dataset: {len(df_complete):,}")
print(f"✅ Ready for surrogate model training with SURGE!")


LOADING CURATED DATASET

✅ Loading curated dataset from pickle file...
   File: /pscratch/sd/a/asvillar/mp288/jobs/batch_16/sdata03_curated.pkl
   Shape: (9891, 1221)
   Memory usage: 92.75 MB

✅ Curated dataset loaded successfully!

CURATED DATASET SUMMARY

Total cases in curated dataset: 9,891
✅ Ready for surrogate model training with SURGE!


## 2. Surrogate Model Training with SURGE

Train surrogate models (Random Forest and MLP) to predict growth rate (gamma) from:
- **Equilibria parameters**: R0, a, kappa, delta
- **Profile characteristics**: q0, q95, qmin, p0


In [3]:
# Set up surrogate model training with SURGE
print("=" * 80)
print("SURROGATE MODEL TRAINING SETUP")
print("=" * 80)

# Verify df_complete is loaded
if 'df_complete' not in locals() or df_complete is None:
    print("❌ Error: df_complete not loaded!")
    print("   Please run the previous cell to load the curated dataset first.")
    raise ValueError("df_complete not available")
else:
    print(f"\n✅ Using curated dataset: {df_complete.shape[0]:,} cases")
    print(f"   Columns: {len(df_complete.columns)}")

# Define input and output variables
print("\n" + "=" * 80)
print("DEFINING INPUT AND OUTPUT VARIABLES")
print("=" * 80)

# Input variables: Equilibria parameters + Profile characteristics
# Equilibria parameters
# Profile characteristics: q0, q95, qmin, p0
input_vars_mapping = {
    'R0': ['eq_R0', 'R0'],        # Major radius [m]
    'a': ['eq_a', 'a'],            # Minor radius [m]
    'kappa': ['eq_kappa', 'kappa'], # Elongation [-]
    'delta': ['eq_delta', 'delta'], # Triangularity [-]
    'q0': ['q0'],                  # q at magnetic axis [-]
    'q95': ['q95'],                # q at 95% flux surface [-]
    'qmin': ['output_qmin', 'qmin'], # Minimum q [-]
    'p0': ['p0']                   # Pressure at magnetic axis [MPa]
}

# Map to actual column names in DataFrame
actual_input_vars = []
for var_name, col_options in input_vars_mapping.items():
    found = False
    for col_name in col_options:
        if col_name in df_complete.columns:
            actual_input_vars.append(col_name)
            print(f"  ✅ {var_name:6s} -> {col_name}")
            found = True
            break
    if not found:
        print(f"  ⚠️  {var_name:6s} -> NOT FOUND (tried: {col_options})")

# Output variable (target for prediction)
output_vars_mapping = {
    'gamma': ['output_gamma', 'gamma']  # Growth rate [s^-1]
}

# Map output variable
actual_output_vars = []
for var_name, col_options in output_vars_mapping.items():
    found = False
    for col_name in col_options:
        if col_name in df_complete.columns:
            actual_output_vars.append(col_name)
            print(f"  ✅ {var_name} -> {col_name}")
            found = True
            break
    if not found:
        print(f"  ⚠️  {var_name} -> NOT FOUND (tried: {col_options})")

print(f"\nInput variables: {len(actual_input_vars)} features")
print(f"  - Equilibria: R0, a, kappa, delta")
print(f"  - Profile characteristics: q0, q95, qmin, p0")
print(f"Output variables: {len(actual_output_vars)} target(s)")
print(f"  - gamma (growth rate)")

# Verify we have all required variables
if len(actual_input_vars) < 8:
    print(f"\n⚠️  Warning: Only found {len(actual_input_vars)}/8 input variables")
if len(actual_output_vars) == 0:
    print(f"\n❌ Error: No output variables found!")
    
# Initialize SURGE engine
print("\n" + "=" * 80)
print("INITIALIZING SURGE ENGINE")
print("=" * 80)

try:
    engine = SurrogateEngine()
    print("✅ SurrogateEngine initialized")
    
    # Load dataset into SURGE
    print("\nLoading dataset into SURGE...")
    engine.load_df_dataset(df_complete, actual_input_vars, actual_output_vars)
    print(f"✅ Dataset loaded: {len(df_complete):,} cases")
    print(f"   Input features: {len(actual_input_vars)}")
    print(f"   Output targets: {len(actual_output_vars)}")
    
except Exception as e:
    print(f"❌ Error initializing SURGE engine: {e}")
    import traceback
    traceback.print_exc()


SURROGATE MODEL TRAINING SETUP

✅ Using curated dataset: 9,891 cases
   Columns: 1221

DEFINING INPUT AND OUTPUT VARIABLES
  ✅ R0     -> eq_R0
  ✅ a      -> eq_a
  ✅ kappa  -> eq_kappa
  ✅ delta  -> eq_delta
  ✅ q0     -> q0
  ✅ q95    -> q95
  ✅ qmin   -> output_qmin
  ✅ p0     -> p0
  ✅ gamma -> output_gamma

Input variables: 8 features
  - Equilibria: R0, a, kappa, delta
  - Profile characteristics: q0, q95, qmin, p0
Output variables: 1 target(s)
  - gamma (growth rate)

INITIALIZING SURGE ENGINE
🚀 Initializing SURGE SurrogateEngine
📊 Features and outputs will be set when loading data
✅ SurrogateEngine initialized

Loading dataset into SURGE...
📥 Loading dataset...
📋 Common output prefix: output_gamma
✅ Dataset loaded: 9891 samples
   📥 Input features: 8 (F=8)
   📤 Output targets: 1 (T=1)
✅ Dataset loaded: 9,891 cases
   Input features: 8
   Output targets: 1


In [4]:
## 3. Data Preprocessing

Split data into train/validation/test sets (70%/10%/20%) and standardize features.


SyntaxError: invalid syntax (40503369.py, line 3)

In [5]:
# Data Preprocessing: Split and Standardize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    engine.X, engine.y, test_size=0.2, random_state=42, shuffle=True
)

# Second split: split remaining 80% into train (70%) and validation (10%)
# test_size = 10/80 = 0.125 to get 10% of total data
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, shuffle=True
)

# Update engine with train/val/test splits
engine.X_train_val = np.vstack([X_train, X_val])
engine.y_train_val = np.vstack([y_train, y_val])
engine.X_test = X_test
engine.y_test = y_test

# Store separate train and val for later use
engine.X_train = X_train
engine.X_val = X_val
engine.y_train = y_train
engine.y_val = y_val

# Standardize data using training set statistics only
# Standardize inputs
engine.x_scaler = StandardScaler()
engine.x_train_sc = engine.x_scaler.fit_transform(X_train)
engine.x_val_sc = engine.x_scaler.transform(X_val)
engine.x_test_sc = engine.x_scaler.transform(X_test)
engine.x_train_val_sc = engine.x_scaler.transform(engine.X_train_val)

# Standardize outputs
engine.y_scaler = StandardScaler()
engine.y_train_sc = engine.y_scaler.fit_transform(y_train)
engine.y_val_sc = engine.y_scaler.transform(y_val)
engine.y_test_sc = engine.y_scaler.transform(y_test)
engine.y_train_val_sc = engine.y_scaler.transform(engine.y_train_val)
print("=" * 80)
print("MODEL INITIALIZATION AND TRAINING")
print("=" * 80)

# Check available models in SURGE
print("\nChecking available models in SURGE...")
try:
    # Try to get model registry
    if hasattr(engine, 'model_registry'):
        print("Available model types:")
        for idx, model_spec in enumerate(engine.model_registry):
            print(f"  {idx}: {model_spec.get('name', 'Unknown')}")
    else:
        print("  Default models: Random Forest (0), Neural Network/MLP (1)")
except:
    print("  Assuming default model order: Random Forest (0), MLP (1)")

# Train Random Forest Regressor (RFR)
print("\n" + "=" * 80)
print("TRAINING RANDOM FOREST REGRESSOR (RFR)")
print("=" * 80)

try:
    # Initialize Random Forest model (index 0)
    # Using named parameters as in demo script
    engine.init_model('random_forest', n_estimators=200, max_depth=15, random_state=42)
    print("✅ Random Forest model initialized")
    
    # Train model
    print("\nTraining Random Forest model...")
    import time
    start_time = time.time()
    engine.train(0)
    train_time_rfr = time.time() - start_time
    print(f"✅ Random Forest training completed in {train_time_rfr:.2f} seconds")
    
    # Get predictions (use predict_output as in demo)
    engine.predict_output(0)
    
    # Print performance metrics
    print(f"\nRandom Forest Performance:")
    if hasattr(engine, 'R2_train_val') and engine.R2_train_val is not None:
        print(f"  Training R²: {engine.R2_train_val:.4f}")
        print(f"  Training MSE: {engine.MSE_train_val:.4e}")
    
    if hasattr(engine, 'R2') and engine.R2 is not None:
        print(f"  Test R²: {engine.R2:.4f}")
        print(f"  Test MSE: {engine.MSE:.4e}")
    
    print(f"  Training time: {train_time_rfr:.2f} seconds")
    
except Exception as e:
    print(f"❌ Error training Random Forest: {e}")
    import traceback
    traceback.print_exc()

# Train Multi-Layer Perceptron (MLP)
print("\n" + "=" * 80)
print("TRAINING MULTI-LAYER PERCEPTRON (MLP)")
print("=" * 80)

try:
    # Initialize MLP/Neural Network model (index 1)
    # Using sklearn.mlp as in demo script
    engine.init_model('sklearn.mlp', hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
    print("✅ MLP model initialized")
    
    # Train model
    print("\nTraining MLP model...")
    start_time = time.time()
    engine.train(1)
    train_time_mlp = time.time() - start_time
    print(f"✅ MLP training completed in {train_time_mlp:.2f} seconds")
    
    # Get predictions (use predict_output as in demo)
    engine.predict_output(1)
    
    # Print performance metrics
    # For model 1, access via model_performance dictionary
    print(f"\nMLP Performance:")
    if hasattr(engine, 'model_performance') and 1 in engine.model_performance:
        perf = engine.model_performance[1]
        if 'R2_train_val' in perf and perf['R2_train_val'] is not None:
            print(f"  Training R²: {perf['R2_train_val']:.4f}")
            print(f"  Training MSE: {perf['MSE_train_val']:.4e}")
        if 'R2' in perf and perf['R2'] is not None:
            print(f"  Test R²: {perf['R2']:.4f}")
            print(f"  Test MSE: {perf['MSE']:.4e}")
    elif hasattr(engine, 'R2_train_val') and engine.R2_train_val is not None:
        # Fallback: use engine-level attributes (may be overwritten by model 1)
        print(f"  Training R²: {engine.R2_train_val:.4f}")
        print(f"  Training MSE: {engine.MSE_train_val:.4e}")
        if hasattr(engine, 'R2') and engine.R2 is not None:
            print(f"  Test R²: {engine.R2:.4f}")
            print(f"  Test MSE: {engine.MSE:.4e}")
    
    print(f"  Training time: {train_time_mlp:.2f} seconds")
    
except Exception as e:
    print(f"❌ Error training MLP: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)
print("MODEL TRAINING SUMMARY")
print("=" * 80)
print("\n✅ Surrogate models trained successfully!")
print("   Models available: Random Forest (index 0), MLP (index 1)")


MODEL INITIALIZATION AND TRAINING

Checking available models in SURGE...
  Default models: Random Forest (0), Neural Network/MLP (1)

TRAINING RANDOM FOREST REGRESSOR (RFR)

🎯 Initializing Model 0: RandomForest (from registry)
✅ RandomForest initialized
✅ Random Forest model initialized

Training Random Forest model...

🎯 Training Model 0: RandomForest
⏱️ Elapsed time: 0.37 seconds
✅ Random Forest training completed in 0.37 seconds

🎯 Predicting outputs - Model 0 (RandomForest)

--- Training Set Results ---
 t_I(avg) = 0.000028 seconds per sample
 MSE_train_val = 0.000055
 R2_train_val = 0.8894

--- Testing Set Results ---
 t_I(avg) = 0.000034 seconds per sample
 MSE = 0.000241
 R2 = 0.5226

Random Forest Performance:
  Training R²: 0.8894
  Training MSE: 5.5145e-05
  Test R²: 0.5226
  Test MSE: 2.4114e-04
  Training time: 0.37 seconds

TRAINING MULTI-LAYER PERCEPTRON (MLP)

🎯 Initializing Model 1: MLP (from registry)
✅ MLP initialized
✅ MLP model initialized

Training MLP model...

🎯 

In [6]:
# Model performance summary and comparison
print("=" * 80)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 80)

# Create comparison table
print(f"\n{'Model':<20} {'Train R²':<15} {'Test R²':<15} {'Test MSE':<15} {'Time (s)':<10}")
print("-" * 80)

# Random Forest performance
rf_train_r2 = engine.R2_train_val if hasattr(engine, 'R2_train_val') else None
rf_test_r2 = engine.R2 if hasattr(engine, 'R2') else None
rf_test_mse = engine.MSE if hasattr(engine, 'MSE') else None

print(f"{'Random Forest':<20}", end="")
print(f"{rf_train_r2:.4f}        " if rf_train_r2 is not None else "N/A           ", end="")
print(f"{rf_test_r2:.4f}        " if rf_test_r2 is not None else "N/A           ", end="")
print(f"{rf_test_mse:.6e}    " if rf_test_mse is not None else "N/A           ", end="")
print(f"{train_time_rfr:.2f}")

# MLP performance
if hasattr(engine, 'model_performance') and 1 in engine.model_performance:
    perf = engine.model_performance[1]
    mlp_train_r2 = perf.get('R2_train_val', None)
    mlp_test_r2 = perf.get('R2', None)
    mlp_test_mse = perf.get('MSE', None)
else:
    mlp_train_r2 = None
    mlp_test_r2 = None
    mlp_test_mse = None

print(f"{'MLP (sklearn)':<20}", end="")
print(f"{mlp_train_r2:.4f}        " if mlp_train_r2 is not None else "N/A           ", end="")
print(f"{mlp_test_r2:.4f}        " if mlp_test_r2 is not None else "N/A           ", end="")
print(f"{mlp_test_mse:.6e}    " if mlp_test_mse is not None else "N/A           ", end="")
print(f"{train_time_mlp:.2f}")

print("-" * 80)

# Feature importance from Random Forest
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE (Random Forest)")
print("=" * 80)

try:
    rf_model = engine.models[0].get_estimator()
    if hasattr(rf_model, 'feature_importances_'):
        importances = rf_model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        print("\nFeature importance scores:")
        print("-" * 80)
        print(f"{'Rank':<6} {'Feature':<20} {'Importance':<15}")
        print("-" * 80)
        
        for rank, idx in enumerate(indices, 1):
            feature_name = actual_input_vars[idx]
            importance = importances[idx]
            print(f"{rank:<6} {feature_name:<20} {importance:.6f}")
        
        print("\n✅ Feature importance analysis complete")
    else:
        print("⚠️  Feature importances not available for this model type")
except Exception as e:
    print(f"⚠️  Could not extract feature importance: {e}")

print("\n✅ Surrogate model training and evaluation complete!")
print("\nModels available:")
print("  - Random Forest (index 0)")
print("  - MLP (index 1)")
print("\nUse engine.predict_output(model_idx) to make predictions on new data.")


MODEL PERFORMANCE SUMMARY

Model                Train R²        Test R²         Test MSE        Time (s)  
--------------------------------------------------------------------------------
Random Forest       0.3528        0.3396        3.335428e-04    0.37
MLP (sklearn)       N/A           N/A           N/A           487.77
--------------------------------------------------------------------------------

FEATURE IMPORTANCE (Random Forest)

Feature importance scores:
--------------------------------------------------------------------------------
Rank   Feature              Importance     
--------------------------------------------------------------------------------
1      output_qmin          0.494105
2      p0                   0.325158
3      q95                  0.061031
4      eq_a                 0.035940
5      eq_kappa             0.023982
6      q0                   0.022663
7      eq_delta             0.019997
8      eq_R0                0.017124

✅ Feature importance analy